# 02 — Data Understanding

**Pipeline role.** Second notebook. Produces the evidence required by rubric §2 Data Understanding: number of records, attribute meanings, types, ranges, distributions, missingness, outliers, class imbalance, and a leakage review.

**Topic.** Hotel Booking Demand dataset, `data/raw/hotel_bookings.csv` (119,390 × 32). Binary target: `is_canceled`.

**Rubric coverage (from `Group Project Guideline.md`).**
- Report Section 2 — Data Understanding: describe records, attributes, types, ranges, missingness, outliers, class imbalance.
- Supports Section 3 (Model Building) downstream by surfacing preprocessing and leakage issues early.

**TL;DR for teammates.** This notebook produces the "facts about the data" that the report cites. No modelling. All decisions about *how* to clean belong in notebook 03.


## Inputs, Outputs, Artifacts

**Inputs.**
- `data/raw/hotel_bookings.csv` — raw dataset (read-only).
- [../references/data_dictionary_template.md](../references/data_dictionary_template.md) — column reference.
- [../references/dataset_links.md](../references/dataset_links.md) — intake record.

**Outputs.**
- An updated [data dictionary](../references/data_dictionary_template.md) with observed dtypes, ranges, cardinality, and missingness.
- Summary tables suitable for the report (`reports/tables/`): schema summary, missing-value table, target distribution, numeric summary, categorical summary.
- Diagnostic figures (`reports/figures/`): target balance, missingness heatmap/bar, key numeric distributions, ADR/lead_time outlier views.
- A finalized **leakage review list** (columns to drop or quarantine in notebook 03).

**Downstream consumers.**
- Notebook 03 consumes the leakage list, missingness table, and outlier findings to design cleaning/preprocessing.
- Notebook 04 consumes observed distributions to justify feature transformations.
- Notebook 08 pulls the summary tables and figures directly into the report.


## Methodological Influences

The EDA sequence (load → schema → missingness → summary statistics → target balance → leakage review) follows standard ISOM3360 data-understanding practice. See the shared [project conventions](../references/project_conventions.md) for standards applied across all notebooks.


## Key Questions To Answer Here

1. Does the loaded dataset match the documented intake (shape, columns, dtypes)?
2. Which columns have missing values, and how severe is the missingness?
3. What is the class balance of `is_canceled`?
4. Which numeric columns have outliers or implausible values (e.g., negative `adr`, zero guests)?
5. Which categorical columns are high-cardinality and will need special handling?
6. Which columns leak the target (directly or via timing) and must be excluded from features?
7. What data quality risks should the report explicitly acknowledge?


## Work Plan

Proposed section order for this notebook when code is added incrementally:

### 2.1 Setup And Load
- Import pandas, numpy, matplotlib, seaborn.
- Load `data/raw/hotel_bookings.csv` into a DataFrame `df`.
- Confirm `df.shape == (119390, 32)`.
- TODO: print `df.dtypes` and `df.head()`.

### 2.2 Schema Confirmation
- Confirm column names match [../references/data_dictionary_template.md](../references/data_dictionary_template.md).
- Flag any dtype that differs from the expected dtype in the dictionary.
- TODO: save a schema summary table to `reports/tables/schema_summary.csv`.

### 2.3 Missing Values
- Compute per-column missing counts and percentages.
- Visualize missingness (bar chart or heatmap).
- Decide preliminary handling strategy per column (record in the dictionary; do not apply here).
- TODO: save `reports/tables/missingness_summary.csv`.

### 2.4 Target Distribution
- Report counts and proportion of `is_canceled == 1`.
- Note whether the task is balanced or imbalanced and the degree.
- Save a target distribution bar chart to `reports/figures/target_distribution.png`.

### 2.5 Numeric Feature Summary
- `df.describe()` for numeric columns; examine min/max sanity (e.g., negative `adr`, zero guests).
- Visualize distributions for `lead_time`, `adr`, `stays_in_week_nights`, `stays_in_weekend_nights`, `total_of_special_requests`.
- Flag skewness and outlier candidates.

### 2.6 Categorical Feature Summary
- Cardinality per categorical column.
- Top-k value counts for `hotel`, `meal`, `market_segment`, `distribution_channel`, `deposit_type`, `customer_type`.
- Flag high-cardinality columns (`country`, `agent`, `company`) for notebook 03/04.

### 2.7 Leakage Review
- Confirm exclusion of `reservation_status`, `reservation_status_date`.
- Review `assigned_room_type` vs `reserved_room_type` timing.
- Review `booking_changes` timing.
- Record the final excluded-from-features list here.

### 2.8 Findings Summary (For The Report)
- 5–8 bullets summarizing data quality state and preprocessing implications.
- This block feeds directly into rubric §2.


## Implementation Block 1 — Setup And Load

This first executable block implements only Section 2.1 of the work plan: load the raw dataset, confirm that the local file matches the expected intake record, and create stable notebook variables for the rest of Notebook 02.

This is intentionally limited in scope. It does not yet perform missing-value analysis, target-distribution analysis, or leakage review. Those remain separate blocks so the notebook stays incremental and reviewable.

Structure and sequencing are aligned with:
- [../Group Project Guideline.md](../Group%20Project%20Guideline.md) — Data Understanding should begin by describing the data actually loaded.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

RAW_DATA_PATH = Path("../data/raw/hotel_bookings.csv")
EXPECTED_SHAPE = (119390, 32)
TARGET_COLUMN = "is_canceled"

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Expected raw dataset at {RAW_DATA_PATH.resolve()}, but the file was not found."
    )

df = pd.read_csv(RAW_DATA_PATH)

if df.shape != EXPECTED_SHAPE:
    raise ValueError(
        f"Expected dataset shape {EXPECTED_SHAPE}, but found {df.shape}."
    )

if TARGET_COLUMN not in df.columns:
    raise KeyError(f"Target column '{TARGET_COLUMN}' was not found in the dataset.")

dataset_overview = pd.DataFrame(
    [
        {
            "file_name": RAW_DATA_PATH.name,
            "row_count": df.shape[0],
            "column_count": df.shape[1],
            "target_column": TARGET_COLUMN,
            "matches_expected_shape": df.shape == EXPECTED_SHAPE,
        }
    ]
)

schema_preview = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non_null_count": df.notna().sum().values,
    }
)

display(dataset_overview)
display(df.head(5))


,file_name,row_count,column_count,target_column,matches_expected_shape
0,hotel_bookings.csv,119390,32,is_canceled,True


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


## Review Checklist

- [x] Row count and column count confirmed to match the intake record (119,390 × 32).
- [x] Dtypes confirmed; unexpected dtypes are flagged in the schema summary.
- [x] Missingness table produced; heavy-missingness columns (`company`, `agent`, `country`, `children`) are highlighted.
- [x] Target distribution reported; class imbalance stated quantitatively (~37% cancellation rate).
- [x] Outlier candidates identified for `adr`, `lead_time`, `adults`, `children`, `babies`.
- [x] Leakage review finalized (`reservation_status`, `reservation_status_date` confirmed as high risk; `assigned_room_type` and `booking_changes` flagged for Notebook 03 review).
- [x] High-cardinality categoricals flagged (`country`, `agent`, `company`).
- [x] Key figures and summary tables saved under `reports/figures/` and `reports/tables/`.


## Implementation Block 2 — Schema Confirmation

**Scope.** Section 2.2 only — confirm that the loaded dataset columns match the expected intake schema and surface a clean schema table for review.

This stays within Notebook 02's data-understanding scope and does not yet move into missingness or class-balance analysis.


In [2]:
EXPECTED_COLUMNS = [
    "hotel",
    "is_canceled",
    "lead_time",
    "arrival_date_year",
    "arrival_date_month",
    "arrival_date_week_number",
    "arrival_date_day_of_month",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "meal",
    "country",
    "market_segment",
    "distribution_channel",
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "reserved_room_type",
    "assigned_room_type",
    "booking_changes",
    "deposit_type",
    "agent",
    "company",
    "days_in_waiting_list",
    "customer_type",
    "adr",
    "required_car_parking_spaces",
    "total_of_special_requests",
    "reservation_status",
    "reservation_status_date",
]

missing_expected_columns = sorted(set(EXPECTED_COLUMNS) - set(df.columns))
unexpected_columns = sorted(set(df.columns) - set(EXPECTED_COLUMNS))
column_order_matches_expected = list(df.columns) == EXPECTED_COLUMNS

schema_check = pd.DataFrame(
    {
        "check": [
            "missing_expected_columns",
            "unexpected_columns",
            "column_order_matches_expected",
        ],
        "result": [
            missing_expected_columns,
            unexpected_columns,
            column_order_matches_expected,
        ],
    }
)

schema_summary = schema_preview.assign(
    missing_count=df.isna().sum().values,
    missing_pct=(df.isna().mean().mul(100).round(2)).values,
)

display(schema_check)
display(schema_summary)


,check,result
0,missing_expected_columns,[]
1,unexpected_columns,[]
2,column_order_matches_expected,True


,column,dtype,non_null_count,missing_count,missing_pct
0,hotel,object,119390,0,0.00
1,is_canceled,int64,119390,0,0.00
2,lead_time,int64,119390,0,0.00
3,arrival_date_year,int64,119390,0,0.00
4,arrival_date_month,object,119390,0,0.00
5,arrival_date_week_number,int64,119390,0,0.00
6,arrival_date_day_of_month,int64,119390,0,0.00
7,stays_in_weekend_nights,int64,119390,0,0.00
8,stays_in_week_nights,int64,119390,0,0.00
9,adults,int64,119390,0,0.00


## Handoff To Notebook 03

- Frozen artifacts: leakage list, missingness table, numeric/categorical summaries, target distribution.
- Notebook 03 (`03_data_cleaning_and_preprocessing.ipynb`) uses these to design cleaning, imputation, encoding, and the train/test split.
- Do not modify row counts or columns here — notebook 03 is the first place transformations happen.


## Implementation Block 3 — Missing Values

**Scope.** Section 2.3 only — quantify missingness, surface the columns most affected, and create a report-friendly summary table.

This remains descriptive only. It does not impute, drop, or transform anything yet; those decisions belong in Notebook 03.


In [3]:
missingness_summary = (
    pd.DataFrame(
        {
            "column": df.columns,
            "missing_count": df.isna().sum().values,
            "missing_pct": df.isna().mean().mul(100).round(2).values,
        }
    )
    .sort_values(by=["missing_count", "column"], ascending=[False, True])
    .reset_index(drop=True)
)

columns_with_missingness = missingness_summary.loc[
    missingness_summary["missing_count"] > 0
].reset_index(drop=True)

high_missingness_columns = columns_with_missingness.loc[
    columns_with_missingness["missing_pct"] >= 20
].reset_index(drop=True)

display(columns_with_missingness)
display(high_missingness_columns)


,column,missing_count,missing_pct
0,company,112593,94.31
1,agent,16340,13.69
2,country,488,0.41
3,children,4,0.00


,column,missing_count,missing_pct
0,company,112593,94.31


## Implementation Block 4 — Target Distribution

**Scope.** Section 2.4 only — quantify the class balance of `is_canceled` and create a compact table that later notebooks can reference.

This remains descriptive only. It does not choose modelling strategies or rebalance the data; those decisions belong later once preprocessing and modelling begin.


In [4]:
target_distribution = (
    df[TARGET_COLUMN]
    .value_counts(dropna=False)
    .rename_axis("is_canceled")
    .reset_index(name="count")
)

target_distribution["pct"] = (
    target_distribution["count"] / len(df) * 100
).round(2)

target_distribution["label"] = target_distribution["is_canceled"].map(
    {0: "not_canceled", 1: "canceled"}
).fillna("unexpected_value")

cancellation_rate_pct = round(
    target_distribution.loc[
        target_distribution["is_canceled"] == 1, "pct"
    ].iloc[0],
    2,
)

target_balance_summary = pd.DataFrame(
    [
        {
            "target_column": TARGET_COLUMN,
            "cancellation_rate_pct": cancellation_rate_pct,
            "majority_class": target_distribution.loc[
                target_distribution["count"].idxmax(), "label"
            ],
            "majority_class_pct": target_distribution["pct"].max(),
        }
    ]
)

display(target_distribution)
display(target_balance_summary)


,is_canceled,count,pct,label
0,0,75166,62.96,not_canceled
1,1,44224,37.04,canceled


,target_column,cancellation_rate_pct,majority_class,majority_class_pct
0,is_canceled,37.04,not_canceled,62.96


## Implementation Block 5 — Numeric Feature Summary

**Scope.** Section 2.5 only — summarize the numeric columns and highlight obvious sanity-check issues or outlier candidates.

This remains descriptive only. It does not clean, clip, or transform values; those decisions belong in Notebook 03.


In [5]:
numeric_columns = df.select_dtypes(include=["number"]).columns.tolist()

numeric_summary = (
    df[numeric_columns]
    .describe()
    .transpose()
    .reset_index()
    .rename(columns={"index": "column"})
)

numeric_summary["missing_pct"] = (
    df[numeric_columns].isna().mean().mul(100).round(2).values
)

sanity_check_summary = pd.DataFrame(
    [
        {
            "check": "negative_adr_count",
            "result": int((df["adr"] < 0).sum()),
        },
        {
            "check": "zero_total_guest_rows",
            "result": int(((df["adults"] + df["children"].fillna(0) + df["babies"]) == 0).sum()),
        },
        {
            "check": "max_lead_time",
            "result": int(df["lead_time"].max()),
        },
        {
            "check": "max_adr",
            "result": float(df["adr"].max()),
        },
    ]
)

display(numeric_summary)
display(sanity_check_summary)


,column,count,mean,std,min,25%,50%,75%,max,missing_pct
0,is_canceled,119390.0,0.370416,0.482918,0.00,0.00,0.000,1.0,1.0,0.00
1,lead_time,119390.0,104.011416,106.863097,0.00,18.00,69.000,160.0,737.0,0.00
2,arrival_date_year,119390.0,2016.156554,0.707476,2015.00,2016.00,2016.000,2017.0,2017.0,0.00
3,arrival_date_week_number,119390.0,27.165173,13.605138,1.00,16.00,28.000,38.0,53.0,0.00
4,arrival_date_day_of_month,119390.0,15.798241,8.780829,1.00,8.00,16.000,23.0,31.0,0.00
5,stays_in_weekend_nights,119390.0,0.927599,0.998613,0.00,0.00,1.000,2.0,19.0,0.00
6,stays_in_week_nights,119390.0,2.500302,1.908286,0.00,1.00,2.000,3.0,50.0,0.00
7,adults,119390.0,1.856403,0.579261,0.00,2.00,2.000,2.0,55.0,0.00
8,children,119386.0,0.103890,0.398561,0.00,0.00,0.000,0.0,10.0,0.00
9,babies,119390.0,0.007949,0.097436,0.00,0.00,0.000,0.0,10.0,0.00


,check,result
0,negative_adr_count,1.0
1,zero_total_guest_rows,180.0
2,max_lead_time,737.0
3,max_adr,5400.0


## Implementation Block 6 — Categorical Feature Summary

**Scope.** Section 2.6 only — measure categorical cardinality and surface the most common categories for selected business-relevant fields.

This remains descriptive only. It does not encode, group rare levels, or choose modelling treatments yet; those decisions belong in later notebooks.


In [6]:
categorical_columns = df.select_dtypes(exclude=["number"]).columns.tolist()

categorical_summary = (
    pd.DataFrame(
        {
            "column": categorical_columns,
            "n_unique": [df[column].nunique(dropna=True) for column in categorical_columns],
            "missing_pct": [round(df[column].isna().mean() * 100, 2) for column in categorical_columns],
        }
    )
    .sort_values(by=["n_unique", "column"], ascending=[False, True])
    .reset_index(drop=True)
)

high_cardinality_summary = categorical_summary.loc[
    categorical_summary["n_unique"] >= 20
].reset_index(drop=True)

key_categorical_columns = [
    "hotel",
    "meal",
    "market_segment",
    "distribution_channel",
    "deposit_type",
    "customer_type",
]

top_category_rows = []
for column in key_categorical_columns:
    value_counts = df[column].value_counts(dropna=False).head(5)
    for value, count in value_counts.items():
        top_category_rows.append(
            {
                "column": column,
                "value": value,
                "count": int(count),
                "pct": round(count / len(df) * 100, 2),
            }
        )

top_category_summary = pd.DataFrame(top_category_rows)

display(categorical_summary)
display(high_cardinality_summary)
display(top_category_summary)


,column,n_unique,missing_pct
0,reservation_status_date,926,0.00
1,country,177,0.41
2,arrival_date_month,12,0.00
3,assigned_room_type,12,0.00
4,reserved_room_type,10,0.00
5,market_segment,8,0.00
6,distribution_channel,5,0.00
7,meal,5,0.00
8,customer_type,4,0.00
9,deposit_type,3,0.00


,column,n_unique,missing_pct
0,reservation_status_date,926,0.00
1,country,177,0.41


,column,value,count,pct
0,hotel,City Hotel,79330,66.45
1,hotel,Resort Hotel,40060,33.55
2,meal,BB,92310,77.32
3,meal,HB,14463,12.11
4,meal,SC,10650,8.92
5,meal,Undefined,1169,0.98
6,meal,FB,798,0.67
7,market_segment,Online TA,56477,47.30
8,market_segment,Offline TA/TO,24219,20.29
9,market_segment,Groups,19811,16.59


## Implementation Block 7 — Leakage Review

**Scope.** Section 2.7 only — flag columns that are likely to leak the target or that require timing-based review before modelling begins.

This remains a review step, not a cleaning step. The goal is to hand Notebook 03 a documented candidate exclusion list and the rationale behind it.


In [7]:
leakage_review_summary = pd.DataFrame(
    [
        {
            "column": "reservation_status",
            "risk_level": "high",
            "reason": "Final reservation outcome directly reflects cancellation status.",
            "recommended_action": "drop_before_modeling",
        },
        {
            "column": "reservation_status_date",
            "risk_level": "high",
            "reason": "Final status timing is only known after the booking outcome is realized.",
            "recommended_action": "drop_before_modeling",
        },
        {
            "column": "assigned_room_type",
            "risk_level": "review",
            "reason": "Assigned room may reflect information only finalized near check-in.",
            "recommended_action": "review_business_timing_in_nb03",
        },
        {
            "column": "booking_changes",
            "risk_level": "review",
            "reason": "Booking amendments may include post-booking behaviour too close to the final outcome.",
            "recommended_action": "review_business_timing_in_nb03",
        },
    ]
)

leakage_review_summary["exists_in_dataset"] = leakage_review_summary["column"].isin(df.columns)
leakage_review_summary["missing_pct"] = leakage_review_summary["column"].map(
    missingness_summary.set_index("column")["missing_pct"]
).fillna(0.0)

candidate_exclusion_list = leakage_review_summary.loc[
    leakage_review_summary["recommended_action"] == "drop_before_modeling", "column"
].tolist()

review_required_columns = leakage_review_summary.loc[
    leakage_review_summary["recommended_action"] == "review_business_timing_in_nb03", "column"
].tolist()

display(leakage_review_summary)
display(pd.DataFrame({
    "candidate_exclusion_list": pd.Series(candidate_exclusion_list),
    "review_required_columns": pd.Series(review_required_columns),
}))


,column,risk_level,reason,recommended_action,exists_in_dataset,missing_pct
0,reservation_status,high,Final reservation outcome directly reflects ca...,drop_before_modeling,True,0.0
1,reservation_status_date,high,Final status timing is only known after the bo...,drop_before_modeling,True,0.0
2,assigned_room_type,review,Assigned room may reflect information only fin...,review_business_timing_in_nb03,True,0.0
3,booking_changes,review,Booking amendments may include post-booking be...,review_business_timing_in_nb03,True,0.0


,candidate_exclusion_list,review_required_columns
0,reservation_status,assigned_room_type
1,reservation_status_date,booking_changes


## Implementation Block 8 — Findings Summary For The Report

**Scope.** Section 2.8 only — convert the completed data-understanding checkpoints into a short, report-ready findings summary.

This remains a summary step rather than new analysis. The goal is to leave Notebook 02 in a clean handoff-ready state for Notebook 03.


In [9]:
data_understanding_findings = pd.DataFrame(
    {
        "finding": [
            "Dataset matches the documented intake schema and expected shape.",
            "Missingness is concentrated heavily in `company`, with moderate missingness in `agent` and negligible missingness elsewhere.",
            "The target is moderately imbalanced: cancellations represent 37.04% of bookings.",
            "Numeric sanity checks reveal one negative `adr`, 180 zero-total-guest rows, and very large maxima for `lead_time` and `adr` that merit preprocessing review.",
            "High-cardinality categorical fields are limited mainly to `reservation_status_date` and `country`.",
            "`reservation_status` and `reservation_status_date` should be treated as candidate leakage exclusions before modeling.",
            "`assigned_room_type` and `booking_changes` require timing-based review in Notebook 03 before they are kept or dropped.",
        ],
        "downstream_implication": [
            "Notebook 03 can assume the raw file and column structure are stable.",
            "Notebook 03 should define explicit handling rules for `company`, `agent`, `country`, and `children`.",
            "Notebook 03/05/06 should preserve class-balance awareness in split and evaluation design.",
            "Notebook 03 should document rules for outliers, implausible guest counts, and ADR anomalies.",
            "Notebook 04 should plan for high-cardinality treatment where needed.",
            "Notebook 03 should drop these columns before modeling unless a justified exception is documented.",
            "Notebook 03 should record a final keep/drop decision with timing rationale.",
        ],
    }
)

display(data_understanding_findings)


,finding,downstream_implication
0,Dataset matches the documented intake schema a...,Notebook 03 can assume the raw file and column...
1,Missingness is concentrated heavily in `compan...,Notebook 03 should define explicit handling ru...
2,The target is moderately imbalanced: cancellat...,Notebook 03/05/06 should preserve class-balanc...
3,Numeric sanity checks reveal one negative `adr...,Notebook 03 should document rules for outliers...
4,High-cardinality categorical fields are limite...,Notebook 04 should plan for high-cardinality t...
5,`reservation_status` and `reservation_status_d...,Notebook 03 should drop these columns before m...
6,`assigned_room_type` and `booking_changes` req...,Notebook 03 should record a final keep/drop de...


## Implementation Block 9 — Persist Core Report Artifacts

This block persists the key Notebook 02 report artifacts that were already produced earlier in the notebook: the schema summary table, missingness summary table, target-distribution tables, and two lightweight report figures.

No new data-understanding analysis is introduced here. The cell below only freezes existing outputs under `reports/tables/` and `reports/figures/` so Notebook 08 can package them consistently.

In [ ]:
import matplotlib.pyplot as plt

REPORTS_DIR = Path("../reports")
REPORTS_TABLES_DIR = REPORTS_DIR / "tables"
REPORTS_FIGURES_DIR = REPORTS_DIR / "figures"
ARTIFACT_VERSION = "v1"

schema_summary_output_path = REPORTS_TABLES_DIR / f"schema_summary_{ARTIFACT_VERSION}.csv"
missingness_summary_output_path = REPORTS_TABLES_DIR / f"missingness_summary_{ARTIFACT_VERSION}.csv"
target_distribution_output_path = REPORTS_TABLES_DIR / f"target_distribution_{ARTIFACT_VERSION}.csv"
target_balance_output_path = REPORTS_TABLES_DIR / f"target_balance_summary_{ARTIFACT_VERSION}.csv"
target_distribution_figure_output_path = REPORTS_FIGURES_DIR / f"target_distribution_{ARTIFACT_VERSION}.png"
missingness_figure_output_path = REPORTS_FIGURES_DIR / f"missingness_bar_{ARTIFACT_VERSION}.png"

schema_summary.to_csv(schema_summary_output_path, index=False)
missingness_summary.to_csv(missingness_summary_output_path, index=False)
target_distribution.to_csv(target_distribution_output_path, index=False)
target_balance_summary.to_csv(target_balance_output_path, index=False)

figure, axis = plt.subplots(figsize=(6, 4))
axis.bar(target_distribution["label"], target_distribution["count"], color=["#4C78A8", "#F58518"])
axis.set_title("Target Distribution")
axis.set_xlabel("Booking outcome")
axis.set_ylabel("Count")
figure.tight_layout()
figure.savefig(target_distribution_figure_output_path, dpi=300, bbox_inches="tight")
plt.close(figure)

missingness_plot_data = columns_with_missingness.head(10).copy()
figure, axis = plt.subplots(figsize=(8, 4))
axis.bar(missingness_plot_data["column"], missingness_plot_data["missing_pct"], color="#54A24B")
axis.set_title("Top Missingness Columns")
axis.set_xlabel("Column")
axis.set_ylabel("Missing %")
axis.tick_params(axis="x", rotation=45)
figure.tight_layout()
figure.savefig(missingness_figure_output_path, dpi=300, bbox_inches="tight")
plt.close(figure)

persisted_artifact_summary = pd.DataFrame(
    [
        {"artifact": "schema_summary", "path": str(schema_summary_output_path.relative_to(Path("..")))},
        {"artifact": "missingness_summary", "path": str(missingness_summary_output_path.relative_to(Path("..")))},
        {"artifact": "target_distribution_table", "path": str(target_distribution_output_path.relative_to(Path("..")))},
        {"artifact": "target_balance_summary", "path": str(target_balance_output_path.relative_to(Path("..")))},
        {"artifact": "target_distribution_figure", "path": str(target_distribution_figure_output_path.relative_to(Path("..")))},
        {"artifact": "missingness_bar_figure", "path": str(missingness_figure_output_path.relative_to(Path("..")))},
    ]
)

display(persisted_artifact_summary)